In [ ]:
from eye_model_3d import MappedDataset
from eye_model_3d.geometry import rotation_3d, pq_to_xy
import numpy as np

In [ ]:
from matplotlib import pyplot as plt

# Load model

In [ ]:
# # Original dataset from Zhao et al.:
# from eye_model_data import ZhaoDataset
# model = ZhaoDataset.load("20240701", convergence=np.radians(5))

# FlyWire mapped dataset:
model = MappedDataset.load("flywire", convergence=np.radians(5))

print(model)

Plot the xyz positions of the ommatidia lenses for the left and right eye:

In [ ]:
fig, ax = plt.subplots(figsize=(2, 2), dpi=300, subplot_kw=dict(projection="3d"))
ax.scatter(*model["left"].xyz_l.T, c="m", lw=0, s=5, alpha=0.5)
ax.scatter(*model["right"].xyz_l.T, c="g", lw=0, s=5, alpha=0.5)
ax.set_aspect("equal")
ax.view_init(10, -15, 0)

Plot the euler angles of the ommatidia sight lines:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(3, 1), dpi=300, subplot_kw=dict(projection="3d"))
for i, ax in enumerate(axes):
    for eye in model.values():
        ang = eye.angles[:, i]  # get corresponding Euler angle
        ax.scatter(*eye.xyz_l.T, c=ang, lw=0, s=1, cmap="bwr", vmin=-np.pi / 2, vmax=np.pi / 2)
    ax.set_aspect("equal")
    ax.view_init(10, -15, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

# Create stimuli

In [ ]:
thetas = np.radians(np.arange(-30, 31, 15))  # azimuths
phis = np.radians(np.arange(-45, 46, 15))    # elevations

xyz = np.column_stack([np.cos(thetas), np.sin(thetas), np.zeros_like(thetas)])

R = rotation_3d(phis, 1)  # rotate around y-axis
xyz = np.einsum("ijk,nj->kni", R, xyz)  # phis, thetas, xyz

distances = np.array([1000, 2000, 4_000])  # in um
xyz = xyz[:, :, None] * distances[..., None]  # thetas, phis, distances, xyz

In [ ]:
print(xyz.shape, ("azimuths", "elevations", "distances", "xyz"))

# Compute column activations

In [ ]:
stimulus_radius = 500  # 1 mm stimulus

u_left = model["left"].column_activation(xyz, stimulus_radius, newaxis=False)
u_right = model["right"].column_activation(xyz, stimulus_radius, newaxis=False)

In [ ]:
print(u_left.shape, ("azimuths", "elevations", "distances", "columns"))

Plot column activation at each distance:

In [ ]:
for i, dist in enumerate(distances):
    fig, axes = plt.subplots(len(phis), len(thetas), figsize=(len(thetas), len(phis)), dpi=300, subplot_kw=dict(projection="3d"))
    fig.suptitle(f"Distance = {dist / 1000} mm")
    for j, row in enumerate(axes):
        for k, ax in enumerate(row):
            ax.scatter(*model["left"].xyz_l.T, c=u_left[j, k, i], lw=0, s=1, alpha=0.5, cmap="inferno")
            ax.scatter(*model["right"].xyz_l.T, c=u_right[j, k, i], lw=0, s=1, alpha=0.5, cmap="inferno")
            ax.set_aspect("equal")
            ax.view_init(10, -15, 0)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_zticks([])
    plt.tight_layout()

# Plot activations in 2D

In [ ]:
# Convert pq hexagonal coordinates to xy cartesian coordinates
xy_l = pq_to_xy(model["left"].pq)
xy_r = pq_to_xy(model["right"].pq, flip=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2, 1), dpi=300)
axes[0].scatter(*xy_l.T, lw=0, s=2, c=u_left[3, 2, 1], cmap="inferno")
axes[1].scatter(*xy_r.T, lw=0, s=2, c=u_right[3, 2, 1], cmap="inferno")
for ax in axes:
    ax.set_aspect("equal")
    ax.axis("off")
axes[0].set_title("Left")
axes[1].set_title("Right")

# Plot columns in Mollweide projection

In [ ]:
proj_l = model["left"].mollweide(distances[1])
proj_r = model["right"].mollweide(distances[1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2, 1), dpi=300, sharey=True)
axes[0].scatter(*proj_l.T, c=u_left[3, 2, 1], cmap="inferno", lw=0, s=2)
axes[1].scatter(*proj_r.T, c=u_right[3, 2, 1], cmap="inferno", lw=0, s=2)
for ax in axes:
    ax.set_aspect("equal")
axes[0].set_title("Left")
axes[1].set_title("Right")

# Column activations for variable stimulus sizes

In [ ]:
stimulus_radii = np.arange(100, 501, 100)

# Create a new axis for each stimulus radius
u_left = model["left"].column_activation(xyz, stimulus_radii, newaxis=True)
u_right = model["right"].column_activation(xyz, stimulus_radii, newaxis=True)

In [ ]:
print(u_left.shape, ("azimuths", "elevations", "distances", "radius", "columns"))

In [ ]:
# Compute radius for constant visual half angle
constant_visual_halfang = np.radians(10)
stimulus_radii = distances * np.sin(constant_visual_halfang)

u_left = model["left"].column_activation(xyz, stimulus_radii, newaxis=False)  # do not create new axis
u_right = model["right"].column_activation(xyz, stimulus_radii, newaxis=False)

In [ ]:
print(u_left.shape, ("azimuths", "elevations", "distances", "columns"))